<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/02_no_supervisado/21_kmeans_gmm.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# K-means y mezclas gaussianas

**Pregunta guía:** ¿Cuándo conviene una asignación dura o probabilística?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


K-means minimiza $\sum_i\|x_i-\mu_{z_i}\|^2$: supone grupos aproximadamente
esféricos de varianza similar y produce una etiqueta dura. Una mezcla
gaussiana modela $p(x)=\sum_k\pi_k\mathcal N(x\mid\mu_k,\Sigma_k)$ y EM
alterna responsabilidades y parámetros; entrega probabilidades.

Simularemos tres poblaciones estelares con covarianzas distintas.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)
parámetros = [
    ([-2, 0], [[0.20, 0.10], [0.10, 1.10]], 260),
    ([1.4, 1.2], [[1.00, -0.55], [-0.55, 0.50]], 330),
    ([2.5, -1.5], [[0.25, 0.0], [0.0, 0.25]], 180),
]
grupos = [rng.multivariate_normal(m, c, n) for m, c, n in parámetros]
X = np.vstack(grupos)
y_real = np.concatenate([np.full(len(g), i) for i, g in enumerate(grupos)])
Xs = StandardScaler().fit_transform(X)


In [ ]:
diagnóstico = []
for k in range(1, 8):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEMILLA).fit(Xs)
    diagnóstico.append(
        {
            "k": k,
            "inercia": km.inertia_,
            "silhouette": silhouette_score(Xs, km.labels_) if k > 1 else np.nan,
        }
    )
diagnóstico = pd.DataFrame(diagnóstico)
display(diagnóstico)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(diagnóstico.k, diagnóstico.inercia, "o-")
axes[0].set(title="Codo", xlabel="k", ylabel="inercia")
axes[1].plot(diagnóstico.k, diagnóstico.silhouette, "o-")
axes[1].set(title="Silhouette", xlabel="k", ylabel="score")
plt.show()


In [ ]:
modelos_gmm, criterios = {}, []
for cov in ["spherical", "diag", "tied", "full"]:
    for k in range(1, 7):
        gmm = GaussianMixture(
            n_components=k, covariance_type=cov, n_init=5, random_state=SEMILLA
        ).fit(Xs)
        modelos_gmm[(cov, k)] = gmm
        criterios.append({"covarianza": cov, "k": k, "BIC": gmm.bic(Xs), "AIC": gmm.aic(Xs)})
criterios = pd.DataFrame(criterios)
display(criterios.sort_values("BIC").head(8))

km = KMeans(n_clusters=3, n_init=30, random_state=SEMILLA).fit(Xs)
clave = criterios.sort_values("BIC").iloc[0][["covarianza", "k"]]
gmm = modelos_gmm[(clave["covarianza"], int(clave["k"]))]
prob = gmm.predict_proba(Xs)
incertidumbre = 1 - prob.max(axis=1)
print("ARI K-means:", adjusted_rand_score(y_real, km.labels_))
print("ARI GMM:", adjusted_rand_score(y_real, gmm.predict(Xs)))
print("Mayor incertidumbre de pertenencia:", incertidumbre.max())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(*Xs.T, c=y_real, s=12, cmap="tab10")
axes[0].set_title("poblaciones simuladas")
axes[1].scatter(*Xs.T, c=km.labels_, s=12, cmap="tab10")
axes[1].set_title("K-means")
im = axes[2].scatter(*Xs.T, c=incertidumbre, s=12, cmap="magma")
axes[2].set_title("incertidumbre GMM")
fig.colorbar(im, ax=axes[2])
plt.show()


**Ejercicios:** compare `full` y `spherical`; transforme una variable a
otra unidad sin escalar; inyecte una cuarta población pequeña; explique
por qué BIC, silhouette y significado astrofísico pueden sugerir números
de grupos distintos.
